# Глава 7: Создание чат-приложений
## Быстрый старт с Microsoft Foundry Models API

Эта записная книжка адаптирована из [репозитория примеров Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), который содержит записные книжки с доступом к сервисам [Azure OpenAI](aoai-assignment.ipynb).

> **Примечание:** GitHub Models будет закрыт в конце июля 2026 года. Эта записная книжка теперь ориентирована на [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), которые предлагают тот же каталог моделей с бесплатным доступом для пробного использования и опыт работы с Azure AI Inference SDK.


# Обзор  
«Большие языковые модели — это функции, которые отображают текст в текст. Получив на вход строку текста, большая языковая модель пытается предсказать, какой текст будет следующим»(1). Этот блокнот «быстрого старта» познакомит пользователей с основными концепциями больших языковых моделей, основными требованиями к пакетам для начала работы с AML, мягким введением в проектирование подсказок и несколькими короткими примерами различных вариантов использования. 


## Содержание  

[Обзор](#overview)  
[Как использовать сервис OpenAI](#how-to-use-openai-service)  
[1. Создание вашего сервиса OpenAI](#1.-creating-your-openai-service)  
[2. Установка](#2.-installation)    
[3. Учетные данные](#3.-credentials)  

[Примеры использования](#use-cases)    
[1. Резюмирование текста](#1.-summarize-text)  
[2. Классификация текста](#2.-classify-text)  
[3. Генерация новых названий продуктов](#3.-generate-new-product-names)  
[4. Тонкая настройка классификатора](#4.fine-tune-a-classifier)  

[Ссылки](#references)


### Создайте ваш первый запрос  
Это короткое упражнение предоставит базовое введение в отправку запросов модели в Microsoft Foundry Models для простой задачи "суммирование".


**Шаги**:  
1. Установите библиотеку `azure-ai-inference` в ваше python-окружение, если вы этого еще не сделали.  
2. Загрузите стандартные вспомогательные библиотеки и настройте учетные данные Microsoft Foundry Models.  
3. Выберите модель для вашей задачи  
4. Создайте простой запрос для модели  
5. Отправьте ваш запрос к API модели!


### 1. Установка `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Импорт вспомогательных библиотек и создание учетных данных


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Поиск подходящей модели  
Модели, такие как GPT-4o и GPT-4o mini, могут понимать и генерировать естественный язык, и доступны в каталоге Microsoft Foundry Models вместе с моделями от Meta, Mistral, Cohere и Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Проектирование подсказок  

"Магия больших языковых моделей заключается в том, что, обучаясь минимизировать ошибку прогнозирования на огромных объемах текста, модели в итоге усваивают концепции, полезные для этих предсказаний. Например, они изучают такие концепции, как"(1):

* как писать слова
* как работает грамматика
* как перефразировать
* как отвечать на вопросы
* как вести разговор
* как писать на многих языках
* как программировать
* и так далее.

#### Как контролировать большую языковую модель  
"Из всех входных данных большой языковой модели самым влиятельным, безусловно, является текстовая подсказка(1).

Большие языковые модели можно побуждать к генерации вывода несколькими способами:

Инструкция: Скажите модели, чего вы хотите
Завершение: Побудьте модель дополнить начало того, что вы хотите
Демонстрация: Покажите модели, чего вы хотите, используя либо:
Несколько примеров в подсказке
Несколько сотен или тысяч примеров в тренировочном наборе для тонкой настройки"



#### Существует три основных рекомендации по созданию подсказок:

**Показывайте и рассказывайте**. Сделайте понятно, чего вы хотите — через инструкции, примеры или их комбинацию. Если вы хотите, чтобы модель отсортировала список элементов в алфавитном порядке или классифицировала абзац по настроению, покажите ей, что именно вы хотите.

**Предоставляйте качественные данные**. Если вы пытаетесь создать классификатор или заставить модель следовать определённому шаблону, убедитесь, что примеров достаточно. Обязательно проверяйте ваши примеры — модель обычно достаточно умна, чтобы распознать базовые орфографические ошибки и дать вам ответ, но также может предположить, что это сделано намеренно, что может повлиять на ответ.

**Проверьте настройки.** Параметры temperature и top_p управляют тем, насколько детерминированным будет ответ модели. Если вы запрашиваете ответ, где есть только один правильный вариант, лучше выставьте эти параметры на низкие значения. Если вы хотите получить более разнообразные ответы, установите их выше. Самая распространённая ошибка — считать эти параметры контролем «сообразительности» или «креативности».


Источник: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Отправить!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Повторите тот же вызов, как сравниваются результаты? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Суммирование текста  
#### Задача  
Суммируйте текст, добавляя «tl;dr:» в конце текстового отрывка. Обратите внимание, как модель понимает, как выполнять ряд задач без дополнительных инструкций. Вы можете экспериментировать с более описательными подсказками, чем tl;dr, чтобы изменить поведение модели и настроить получаемое суммирование(3).  

Недавние исследования показали существенные улучшения во многих задачах и бенчмарках NLP благодаря предварительному обучению на большом корпусе текста с последующей тонкой настройкой на конкретной задаче. Хотя архитектура обычно универсальна для задач, этот метод все равно требует специализированных наборов данных для тонкой настройки, содержащих тысячи или десятки тысяч примеров. В отличие от этого, люди, как правило, могут выполнять новую языковую задачу, имея всего несколько примеров или простые инструкции — то, с чем современные системы NLP по-прежнему в значительной степени испытывают трудности. Здесь мы показываем, что увеличение масштабов языковых моделей значительно улучшает универсальную, малообразцовую производительность, иногда даже достигая конкурентоспособности с предыдущими передовыми методами тонкой настройки.  



Tl;dr


# Упражнения для нескольких вариантов использования  
1. Суммирование текста  
2. Классификация текста  
3. Создание новых названий продуктов


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Классифицировать текст  
#### Задача  
Классифицируйте элементы по категориям, предоставленным во время вывода модели. В следующем примере мы указываем и категории, и текст для классификации в подсказке (*playground_reference). 

Запрос клиента: Здравствуйте, одна из клавиш на клавиатуре моего ноутбука недавно сломалась, и мне потребуется замена:

Классифицированная категория:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Создание новых названий продуктов
#### Задача
Создайте названия продуктов на основе примерных слов. Здесь мы включаем в подсказку информацию о продукте, для которого собираемся генерировать названия. Мы также приводим подобный пример, чтобы показать желаемый образец. Кроме того, мы установили высокое значение температуры, чтобы увеличить случайность и получить более инновационные ответы.

Описание продукта: домашний миксер для молочных коктейлей
Ключевые слова: быстрый, полезный, компактный.
Названия продуктов: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Описание продукта: пара обуви, которая подходит для любого размера ноги.
Ключевые слова: адаптивный, подходящий, универсальный.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Ссылки  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Портал Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Лучшие практики дообучения моделей GPT для классификации текста](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для дополнительной помощи  
[Команда коммерциализации OpenAI](AzureOpenAITeam@microsoft.com) 


# Участники
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
